# 1. Auditoría de Datos: Tabla `orders`

Se realizó un perfilado inicial del archivo `olist_orders_dataset.csv` utilizando la librería Pandas para definir la arquitectura de la tabla en PostgreSQL.

## Hallazgos Técnicos:
* **Volumen de Datos:** 99,441 registros
* **Integridad de Columnas:** Se identificaron **8 columnas** con 3 columnas (fechas) de valores no nulos.

### Análisis de Cardinalidad:
* **`order_id`:** Posee 99,441 valores únicos, confirmando su aptitud como **Primary Key**.
* **`customer_id`:** Posee 99,441 valores unicos lo que indica que a cada compra se le asigna un id aunque se trate del mismo cliente.

In [1]:
import pandas as pd

df_orders = pd.read_csv('../data/olist_orders_dataset.csv')

data_quality = pd.DataFrame({
    'Tipo': df_orders.dtypes,
    'No Nulos': df_orders.count(),
    'Nulos': df_orders.isnull().sum(),
    'Valores Unicos': df_orders.nunique(),
    'Duplicados': len(df_orders) - df_orders.nunique()
})

data_quality

,Tipo,No Nulos,Nulos,Valores Unicos,Duplicados
order_id,object,99441,0,99441,0
customer_id,object,99441,0,99441,0
order_status,object,99441,0,8,99433
order_purchase_timestamp,object,99441,0,98875,566
order_approved_at,object,99281,160,90733,8708
order_delivered_carrier_date,object,97658,1783,81018,18423
order_delivered_customer_date,object,96476,2965,95664,3777
order_estimated_delivery_date,object,99441,0,459,98982


## 1.2. Diagnóstico de Integridad en columna "Status"

Se realizó un análisis de consistencia entre el estado logístico del pedido y la presencia de la fecha de entrega al cliente

* **Consistencia lógica en estados intermedios:** `aprobed`, `created`, `shipped`, `processing`, `invoiced` y `unavailable` presentan un 100% de nulos en la fecha de entrega. Esto es correcto, ya que representan pedidos que aún no han completado el ciclo.
* **Anomalía en Pedidos Entregados:** En `delivered` se detectaron **8 registros** marcados como "entregados" que carecen de marca de tiempo (NaN), representando un error de integridad.
* **Casos de cancelacion post-entrega:** Existen **6 registros** en estado `canceled` que poseen fecha de entrega. Esto sugiere cancelaciones procesadas administrativamente posterior a que el producto llegara fisicamente al cliente.

In [4]:
##############################################################################################################

# AGRUPAR POR ESTADO Y MOSTRAR COLUMNAS SIN Y CON FECHA DE ENTREGA AL CLIENTE

##############################################################################################################

data_status = df_orders.groupby('order_status')['order_delivered_customer_date'].agg(
    Total_Pedidos = 'size',
    Fecha_Faltante = lambda x: x.isnull().sum(),
    Posee_fecha = 'count'
)

data_status['%_Nulos'] = (data_status['Fecha_Faltante'] / data_status['Total_Pedidos']) * 100
data_status['%_Nulos'] = data_status['%_Nulos'].round(4)

data_status


,Total_Pedidos,Fecha_Faltante,Posee_fecha,%_Nulos
order_status,,,,
approved,2,2,0,100.0000
canceled,625,619,6,99.0400
created,5,5,0,100.0000
delivered,96478,8,96470,0.0083
invoiced,314,314,0,100.0000
processing,301,301,0,100.0000
shipped,1107,1107,0,100.0000
unavailable,609,609,0,100.0000
